In [1]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/research-engine
!pip install -q rank_bm25
!ls *.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/research-engine
day3_hybrid_retrieval.py  day4_rerank.py


In [2]:
# 2. Quick check
!python day4_rerank.py --limit 200 --title
!python day4_rerank.py --limit 200 --reranker BAAI/bge-reranker-v2-m3
!python day4_rerank.py --limit 200 --reranker BAAI/bge-reranker-v2-m3 --title

28541 chunks | 200 questions (split=dev, scope=paper)
Loading weights: 100% 199/199 [00:00<00:00, 4318.77it/s]
Loading cached embeddings: cache/emb_BAAI_bge-small-en-v1.5_28541.npy
Loading weights: 100% 201/201 [00:00<00:00, 15653.88it/s]
Batches: 100% 54/54 [01:14<00:00,  1.38s/it]
hybrid_rrf {'hit@1': 0.345, 'hit@3': 0.64, 'hit@5': 0.765, 'hit@10': 0.94, 'ev_recall@1': 0.2527, 'ev_recall@3': 0.5136, 'ev_recall@5': 0.6461, 'ev_recall@10': 0.8869, 'mrr@10': 0.5276}
hybrid+rerank {'hit@1': 0.115, 'hit@3': 0.435, 'hit@5': 0.66, 'hit@10': 0.9, 'ev_recall@1': 0.0892, 'ev_recall@3': 0.3281, 'ev_recall@5': 0.5294, 'ev_recall@10': 0.8344, 'mrr@10': 0.3368}
hybrid+rerank_rrf {'hit@1': 0.285, 'hit@3': 0.58, 'hit@5': 0.775, 'hit@10': 0.94, 'ev_recall@1': 0.1971, 'ev_recall@3': 0.4565, 'ev_recall@5': 0.6536, 'ev_recall@10': 0.8779, 'mrr@10': 0.4735}

Section type at rank 1 (share of questions)  vs  share of gold evidence:
  experiments    gold=0.31  hybrid_rrf=0.24  hybrid+rerank=0.08  hybrid+rer

In [3]:
import json, glob
rows = []
for f in sorted(glob.glob("results/day4_rerank_dev_paper_*.json")):
    d = json.load(open(f))
    tag = d["reranker"].split("/")[-1] + (" +title" if d.get("title") else "")
    for name, r in d["results"].items():
        rows.append((tag, name, r["hit@1"], r["ev_recall@5"], r["ev_recall@10"], r["mrr@10"], d["rerank_ms_per_query"]))
print(f"{'reranker':<28}{'system':<20}{'hit@1':>7}{'R@5':>7}{'R@10':>7}{'MRR':>7}{'ms/q':>7}")
for t, n, h, r5, r10, m, ms in rows:
    print(f"{t:<28}{n:<20}{h:>7.3f}{r5:>7.3f}{r10:>7.3f}{m:>7.3f}{ms:>7.0f}")

reranker                    system                hit@1    R@5   R@10    MRR   ms/q
bge-reranker-base +title    hybrid_rrf            0.345  0.646  0.887  0.528    375
bge-reranker-base +title    hybrid+rerank         0.115  0.529  0.834  0.337    375
bge-reranker-base +title    hybrid+rerank_rrf     0.285  0.654  0.878  0.473    375
bge-reranker-v2-m3          hybrid_rrf            0.345  0.646  0.887  0.528   1196
bge-reranker-v2-m3          hybrid+rerank         0.345  0.741  0.929  0.555   1196
bge-reranker-v2-m3          hybrid+rerank_rrf     0.380  0.713  0.935  0.561   1196
bge-reranker-v2-m3 +title   hybrid_rrf            0.345  0.646  0.887  0.528   1277
bge-reranker-v2-m3 +title   hybrid+rerank         0.190  0.624  0.858  0.410   1277
bge-reranker-v2-m3 +title   hybrid+rerank_rrf     0.335  0.670  0.889  0.519   1277


In [4]:
!python day4_rerank.py --scope paper --reranker BAAI/bge-reranker-v2-m3

28541 chunks | 887 questions (split=dev, scope=paper)
Loading weights: 100% 199/199 [00:00<00:00, 4403.55it/s]
Loading cached embeddings: cache/emb_BAAI_bge-small-en-v1.5_28541.npy
Loading weights: 100% 393/393 [00:00<00:00, 4509.73it/s]
Batches: 100% 237/237 [18:24<00:00,  4.66s/it]
hybrid_rrf {'hit@1': 0.345, 'hit@3': 0.6494, 'hit@5': 0.805, 'hit@10': 0.9481, 'ev_recall@1': 0.2707, 'ev_recall@3': 0.5439, 'ev_recall@5': 0.7099, 'ev_recall@10': 0.9043, 'mrr@10': 0.532}
hybrid+rerank {'hit@1': 0.3551, 'hit@3': 0.6832, 'hit@5': 0.8377, 'hit@10': 0.9662, 'ev_recall@1': 0.29, 'ev_recall@3': 0.5844, 'ev_recall@5': 0.747, 'ev_recall@10': 0.9345, 'mrr@10': 0.5494}
hybrid+rerank_rrf {'hit@1': 0.3878, 'hit@3': 0.7035, 'hit@5': 0.8365, 'hit@10': 0.9651, 'ev_recall@1': 0.3095, 'ev_recall@3': 0.5933, 'ev_recall@5': 0.748, 'ev_recall@10': 0.9337, 'mrr@10': 0.5709}

Section type at rank 1 (share of questions)  vs  share of gold evidence:
  experiments    gold=0.33  hybrid_rrf=0.26  hybrid+rerank=0.2

In [5]:
# reranks and saves the scores to cache/
!python day4_rerank.py --reranker BAAI/bge-reranker-v2-m3 --fp16

28541 chunks | 887 questions (split=dev, scope=paper)
Loading weights: 100% 199/199 [00:00<00:00, 16556.57it/s]
Loading cached embeddings: cache/emb_BAAI_bge-small-en-v1.5_28541.npy
Loading weights: 100% 393/393 [00:00<00:00, 4158.15it/s]
Batches: 100% 237/237 [03:47<00:00,  1.04it/s]
hybrid_rrf {'hit@1': 0.345, 'hit@3': 0.6494, 'hit@5': 0.805, 'hit@10': 0.9481, 'ev_recall@1': 0.2707, 'ev_recall@3': 0.5439, 'ev_recall@5': 0.7099, 'ev_recall@10': 0.9043, 'mrr@10': 0.532}
hybrid+rerank {'hit@1': 0.3563, 'hit@3': 0.6832, 'hit@5': 0.8377, 'hit@10': 0.9662, 'ev_recall@1': 0.2904, 'ev_recall@3': 0.5844, 'ev_recall@5': 0.7476, 'ev_recall@10': 0.9339, 'mrr@10': 0.5502}
hybrid+rerank_rrf {'hit@1': 0.3878, 'hit@3': 0.7035, 'hit@5': 0.8365, 'hit@10': 0.9651, 'ev_recall@1': 0.3095, 'ev_recall@3': 0.5939, 'ev_recall@5': 0.748, 'ev_recall@10': 0.9337, 'mrr@10': 0.5706}

Section type at rank 1 (share of questions)  vs  share of gold evidence:
  experiments    gold=0.33  hybrid_rrf=0.26  hybrid+rerank

In [6]:
# reuses the cached scores
!python day4_rerank.py --reranker BAAI/bge-reranker-v2-m3 --fp16 --exclude-sections abstract

28541 chunks | 887 questions (split=dev, scope=paper)
Loading weights: 100% 199/199 [00:00<00:00, 4504.07it/s]
Loading cached embeddings: cache/emb_BAAI_bge-small-en-v1.5_28541.npy
Loading cached reranker scores: cache/rerank_BAAI_bge-reranker-v2-m3_dev_paper_top30_n887.npy
Excluding section types from rankings: ['abstract']
hybrid_rrf {'hit@1': 0.3777, 'hit@3': 0.6877, 'hit@5': 0.8309, 'hit@10': 0.9583, 'ev_recall@1': 0.2976, 'ev_recall@3': 0.5796, 'ev_recall@5': 0.7474, 'ev_recall@10': 0.9217, 'mrr@10': 0.5638}
hybrid+rerank {'hit@1': 0.4656, 'hit@3': 0.7418, 'hit@5': 0.8771, 'hit@10': 0.9707, 'ev_recall@1': 0.3697, 'ev_recall@3': 0.6396, 'ev_recall@5': 0.7931, 'ev_recall@10': 0.9429, 'mrr@10': 0.6309}
hybrid+rerank_rrf {'hit@1': 0.425, 'hit@3': 0.7475, 'hit@5': 0.8737, 'hit@10': 0.9729, 'ev_recall@1': 0.3345, 'ev_recall@3': 0.6399, 'ev_recall@5': 0.7927, 'ev_recall@10': 0.9467, 'mrr@10': 0.6105}

Section type at rank 1 (share of questions)  vs  share of gold evidence:
  experiments 

In [7]:
!python day4_rerank.py --split test --reranker BAAI/bge-reranker-v2-m3 --fp16 --exclude-sections abstract

28541 chunks | 1309 questions (split=test, scope=paper)
Loading weights: 100% 199/199 [00:00<00:00, 4905.76it/s]
Loading cached embeddings: cache/emb_BAAI_bge-small-en-v1.5_28541.npy
Loading weights: 100% 393/393 [00:00<00:00, 4889.16it/s]
Batches: 100% 353/353 [05:41<00:00,  1.03it/s]
Excluding section types from rankings: ['abstract']
hybrid_rrf {'hit@1': 0.4072, 'hit@3': 0.7189, 'hit@5': 0.8419, 'hit@10': 0.9672, 'ev_recall@1': 0.296, 'ev_recall@3': 0.5815, 'ev_recall@5': 0.7284, 'ev_recall@10': 0.9131, 'mrr@10': 0.5903}
hybrid+rerank {'hit@1': 0.5218, 'hit@3': 0.796, 'hit@5': 0.906, 'hit@10': 0.9817, 'ev_recall@1': 0.3923, 'ev_recall@3': 0.6589, 'ev_recall@5': 0.7953, 'ev_recall@10': 0.9412, 'mrr@10': 0.6772}
hybrid+rerank_rrf {'hit@1': 0.4729, 'hit@3': 0.7807, 'hit@5': 0.8892, 'hit@10': 0.9786, 'ev_recall@1': 0.3464, 'ev_recall@3': 0.6427, 'ev_recall@5': 0.7848, 'ev_recall@10': 0.937, 'mrr@10': 0.6472}

Section type at rank 1 (share of questions)  vs  share of gold evidence:
  exp